# Baseline vs. авторски модел – прогнозиране на цена за нощувка (StayOra)

Тетрадка за Google Colab. Сравнява тривиален модел, класически модел и авторски модел по MAE и RMSE.
Авторският модел се различава от базовия само по един инженеринг признак: `stars * log1p(review_count)`.

## 1. Инсталация и импорти

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split

RANDOM_STATE = 42


## 2. Генериране на синтетичен dataset (StayOra-подобен)

In [ ]:
DATA_MODE = "synthetic"  # Set to "csv" to use an empirical export.
CSV_PATH = "stayora_prices.csv"
CURRENCY = "EUR"  # Confirm and set this to the currency used in the CSV.

required_columns = {"stars", "review_count", "price_per_night"}

if DATA_MODE == "csv":
    df = pd.read_csv(CSV_PATH)
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(f"CSV is missing required columns: {sorted(missing_columns)}")
    df = df[["stars", "review_count", "price_per_night"]].copy()
    print(f"Empirical dataset loaded: {len(df)} rows from {CSV_PATH} ({CURRENCY})")
elif DATA_MODE == "synthetic":
    rng = np.random.default_rng(42)
    sample_count = 1200
    stars = rng.choice([1, 2, 3, 4, 5], size=sample_count, p=[0.05, 0.1, 0.25, 0.4, 0.2])
    reviews = rng.poisson(lam=80, size=sample_count).clip(0, 600)
    engineered_signal = stars * np.log1p(reviews)
    price = 40 + 28 * stars + 0.12 * reviews + 2.5 * engineered_signal
    price += rng.normal(0, 12, size=sample_count)
    df = pd.DataFrame({
        "stars": stars,
        "review_count": reviews,
        "price_per_night": np.clip(price, 15, None).round(2),
    })
    print("SYNTHETIC DEMONSTRATION ONLY: do not report these metrics as StayOra results.")
else:
    raise ValueError("DATA_MODE must be either 'csv' or 'synthetic'.")

if df.empty:
    raise ValueError("The dataset is empty.")
if df[list(required_columns)].isna().any().any():
    raise ValueError("Required columns contain missing values; clean them before training.")
if (df["review_count"] < 0).any() or (df["stars"] < 0).any():
    raise ValueError("stars and review_count must be non-negative.")

print("Rows:", len(df))
df.head()

### Описателна статистика

In [ ]:
df.describe(include="all")

## 3. Feature engineering (само едно добавено нещо за авторския модел)

In [ ]:
df["engineered_stars_reviews"] = df["stars"] * np.log1p(df["review_count"])
features_base = ["stars", "review_count"]
features_authored = features_base + ["engineered_stars_reviews"]
X_base = df[features_base].to_numpy()
X_authored = df[features_authored].to_numpy()
y = df["price_per_night"].to_numpy(dtype=float)


## 4. Train / Test split (80/20, seed=42)

In [ ]:
indices = np.arange(len(df))
train_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=RANDOM_STATE
)
Xb_train, Xb_test = X_base[train_idx], X_base[test_idx]
Xa_train, Xa_test = X_authored[train_idx], X_authored[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train rows: {len(train_idx)} | Test rows: {len(test_idx)}")

## 5. Модели

In [ ]:
models = {
    "Dummy (mean)": (DummyRegressor(strategy="mean"), Xb_train, Xb_test),
    "Linear regression": (LinearRegression(), Xb_train, Xb_test),
    "RF baseline (2 features)": (
        RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=1),
        Xb_train,
        Xb_test,
    ),
    "RF authored (3 features)": (
        RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=1),
        Xa_train,
        Xa_test,
    ),
}

fitted_models = {}
predictions = {}
for name, (model, X_train, X_test) in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    predictions[name] = model.predict(X_test)


## 6. Метрики (MAE, RMSE, R2)

In [ ]:
def metrics(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {"model": name, "MAE": mae, "RMSE": rmse, "R2": r2}

results = [
    metrics(y_test, predictions[name], name)
    for name in models
]
results_df = pd.DataFrame(results).set_index("model")
display(results_df)


## 7. Намаление на грешката (vs. Dummy)

In [ ]:
results_df[["MAE", "RMSE"]].plot(kind="bar", figsize=(9, 4), title="Model comparison")
plt.ylabel(f"Error ({CURRENCY})")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 8. 5-fold CV и paired test върху hold-out

MAE от 5-fold CV се показва описателно; fold-овете не се третират като независими наблюдения за t-тест. Статистическата проверка сравнява попарните абсолютни грешки върху отделения 20% тестов набор. Изисква се едностранна paired permutation проверка при α = 0.05, 95% bootstrap интервал на разликата и поне 5% относително намаление на MAE спрямо Dummy. Този анализ има смисъл за реален CSV набор; в synthetic режим е само демонстрация на кода.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for fold, (train_idx, valid_idx) in enumerate(kf.split(X_base), start=1):
    y_train_fold, y_valid_fold = y[train_idx], y[valid_idx]
    fold_models = {
        "Dummy (mean)": (DummyRegressor(strategy="mean"), X_base[train_idx], X_base[valid_idx]),
        "RF baseline (2 features)": (
            RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=1),
            X_base[train_idx], X_base[valid_idx],
        ),
        "RF authored (3 features)": (
            RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=1),
            X_authored[train_idx], X_authored[valid_idx],
        ),
    }
    for name, (model, X_train_fold, X_valid_fold) in fold_models.items():
        model.fit(X_train_fold, y_train_fold)
        cv_rows.append({
            "fold": fold,
            "model": name,
            "MAE": mean_absolute_error(y_valid_fold, model.predict(X_valid_fold)),
        })

cv_results = pd.DataFrame(cv_rows)
display(cv_results.pivot(index="fold", columns="model", values="MAE"))
print("Mean 5-fold CV MAE (descriptive only):")
display(cv_results.groupby("model")["MAE"].agg(["mean", "std"]))

# Paired inference uses one absolute-error pair per held-out test row, not five fold means.
error_dummy = np.abs(y_test - predictions["Dummy (mean)"])
error_authored = np.abs(y_test - predictions["RF authored (3 features)"])
paired_differences = error_authored - error_dummy
relative_reduction = 1 - error_authored.mean() / error_dummy.mean()

bootstrap_rng = np.random.default_rng(RANDOM_STATE)
bootstrap_means = np.array([
    bootstrap_rng.choice(paired_differences, size=len(paired_differences), replace=True).mean()
    for _ in range(10000)
])
ci_low, ci_high = np.percentile(bootstrap_means, [2.5, 97.5])
permutation = stats.permutation_test(
    (error_authored, error_dummy),
    statistic=lambda authored, dummy: np.mean(authored - dummy),
    permutation_type="samples",
    alternative="less",
    n_resamples=9999,
    rng=np.random.default_rng(RANDOM_STATE),
)

print(f"Hold-out MAE difference (authored - dummy): {paired_differences.mean():.3f} {CURRENCY}")
print(f"Paired bootstrap 95% CI: [{ci_low:.3f}, {ci_high:.3f}] {CURRENCY}")
print(f"One-sided paired permutation p-value: {permutation.pvalue:.5f}")
print(f"Relative MAE reduction vs Dummy: {relative_reduction:.1%}")

if DATA_MODE == "csv":
    supported = ci_high < 0 and permutation.pvalue < 0.05 and relative_reduction >= 0.05
    print("Predefined evidence criterion met:", supported)
else:
    print("Inferential statistics above are a pipeline demonstration only; rerun with empirical CSV data.")

## 9. Заключение

In [ ]:
if DATA_MODE == "synthetic":
    print("Do not report these metrics as empirical StayOra results.")
    print("Set DATA_MODE = 'csv' and provide a verified export before drawing conclusions.")
else:
    print(f"Evaluated {len(df)} empirical records from {CSV_PATH}.")
    print("Report the source, extraction date, currency, cleaning rules, and limitations.")
print("Compare only the predefined paired-error criterion; do not infer causality.")